# 03 Portfolio Backtest

Notebook n?y ch?y backtest cho **m?t danh m?c symbol** b?ng c?ng chi?n l??c Combo.
M?c ti?u l? xem equity t?ng, metrics portfolio v? drill-down theo t?ng symbol.


In [ ]:
# Bootstrap: add repo root + core_python to sys.path
import sys
from pathlib import Path

def _find_root(start: Path, marker: str = 'config.py') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError(f'Could not locate repo root containing {marker!r}')

ROOT = _find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)
print('CORE =', CORE)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.style.use('dark_background')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

from strategies.combo.config import SYMBOLS, summary as strategy_summary
from strategies.combo.portfolio.backtest import run_portfolio_backtest

print(strategy_summary())
print('Available symbols:', ', '.join(SYMBOLS.keys()))


In [ ]:
SYMBOLS_TO_RUN = ['US30', 'US500', 'DE40', 'GOLD', 'BTCUSD']
ACCOUNT_MODE = 'standard'
INITIAL_BALANCE = 100_000.0
DATE_FROM = '2023-01-01'
DATE_TO = None
MAX_BARS = 30000

SYMBOL_OVERRIDES = {
    # 'US30': {'x': 10.0, 'ktp': 2.3, 'ma_period': 20, 'trailing_activation': 1.0},
}


In [ ]:
portfolio = run_portfolio_backtest(
    symbol_keys=SYMBOLS_TO_RUN,
    initial_balance=INITIAL_BALANCE,
    account_mode=ACCOUNT_MODE,
    date_from=DATE_FROM,
    date_to=DATE_TO,
    symbol_overrides=SYMBOL_OVERRIDES or None,
    max_bars=MAX_BARS,
)

print('Mode =', portfolio.account_mode)
print('Symbols =', portfolio.symbol_keys)


In [ ]:
portfolio_metrics = pd.Series({k: v for k, v in portfolio.metrics.items() if k != 'monthly_pnl_table'})
display(portfolio_metrics.to_frame('value'))

per_symbol_rows = []
for sym, result in portfolio.symbol_results.items():
    row = {'symbol': sym, **{k: v for k, v in result.metrics.items() if k != 'monthly_pnl_table'}}
    per_symbol_rows.append(row)

per_symbol_df = pd.DataFrame(per_symbol_rows).sort_values('sharpe', ascending=False, ignore_index=True)
display(per_symbol_df)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=False)

portfolio.combined_equity.plot(ax=axes[0], color='#00D4FF', lw=2, title='Combined portfolio equity')
axes[0].grid(alpha=0.3)

portfolio.equity_frame.plot(ax=axes[1], lw=1.2, title='Per-symbol equity curves')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
trades_df = pd.DataFrame(portfolio.trades)
if not trades_df.empty:
    display(trades_df.tail(50))
else:
    print('No portfolio trades.')
